# Analyzing molecular dynamics simulations

### Overview

In this exercise, we explore several standard methods used to analyze molecular dynamics (MD) simulations, with the goal of extracting **structural, dynamical, and functional insight** from raw trajectory data.

Starting from replica exchange simulation generated in Exercise 10, we perform sampling importance resampling and alignment. Unlike in Exercise 10, the alignment will be based on alpha carbons of the transmembrane helices. Then we will analyze the simulation using complementary approaches that address different questions:

- **Root Mean Square Fluctuation (RMSF)** to identify which regions of the protein are most flexible.
- **Principal Component Analysis (PCA)** to uncover dominant collective motions in the protein.
- **PCA-based animations and porcupine plots** to interpret these motions structurally.
- **Clustering in PCA space** to assess whether the simulation samples distinct conformational states.
- **Medoid extraction and visualization** to obtain representative structures from each cluster.
- **Protein–ligand contact analysis** to relate conformational dynamics to binding interactions.

Together, these analyses illustrate how MD trajectories can be transformed from large sets of atomic coordinates into interpretable descriptions of protein motion, conformational states, and structure–function relationships.

Parts of this exercise were adapted from [talktorial 20 of teachopencadd](https://github.com/volkamerlab/teachopencadd/tree/master/teachopencadd/talktorials/T020_md_analysis). This exercise was designed to be run on the SDSC Expanse high performance computing resource.

Assuming that you are reading this Notebook on Github, you will need to download it to your account on SDSC Expanse. To do that (and to load additional Exercise 11 scripts), log onto the Expanse User Portal, select *Shell*, and paste the following commands into the terminal:

```bash
mkdir -p ~/exercises
cd ~/exercises
curl -L -o 11-MD_Analysis.ipynb https://raw.githubusercontent.com/daveminh/Chem456-2026S/refs/heads/main/exercises/11-MD_Analysis.ipynb
curl -L -o 11-SIRS.job https://raw.githubusercontent.com/daveminh/Chem456-2026S/refs/heads/main/exercises/11-SIRS.job
curl -L -o 11-SIRS.py https://raw.githubusercontent.com/daveminh/Chem456-2026S/refs/heads/main/exercises/11-SIRS.py
```

Before running this notebook, you should generate a sampling importance resampling (SIRS) trajectory from Exercise 10. First, you need to edit `11-SIRS.job` (e.g. with nano) to change the `MM_SYSTEM` variable. Then you can run it with `sbatch 11-SIRS.job`. After the job is complete, this notebook is ready to run.

For this notebook, we will use the `shared` partition and `visualization` conda environment.

```bash
/cm/shared/apps/sdsc/galyleo/galyleo launch --account iit130 --partition shared --cpus 4 --memory 8 --time-limit 02:00:00 --interface lab --conda-env visualization --conda-init "$HOME/miniconda3/etc/profile.d/conda.sh"
```

The exercise will be graded based on submitting your answers to the questions after ```-->``` on Canvas.

## Part 0 – Installs and imports

In [ ]:
! mamba install -c conda-forge hdbscan -y

In [ ]:
%matplotlib inline

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysis.analysis.distances import distance_array

from sklearn.decomposition import PCA
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster

import nglview as nv
import ipywidgets as widgets

## Part 1 – Load the aligned SIRS trajectory

In this exercise, we analyze a **pre-aligned molecular dynamics (MD) trajectory** generated in `11-SIRS.py`. The trajectory has already been processed to align to the transmembrane helicies, so that the remaining motion reflects **internal conformational changes** rather than rigid-body drift.

We focus on the **Cα atoms** of the protein backbone. This provides a compact yet informative representation of protein motion and greatly reduces the dimensionality of the data.

In [ ]:
MM_system = 'MSX-2'  # change as needed

base_dir = os.path.expanduser(f'~/exercises/11/{MM_system}')
pdb_out  = os.path.join(base_dir, f'{MM_system}_SIRS_reference.pdb')
traj_out = os.path.join(base_dir, f'{MM_system}_SIRS_aligned.dcd')

u = mda.Universe(pdb_out, traj_out)
protein = u.select_atoms('protein')
ligand  = u.select_atoms('resname UNK')
ca = u.select_atoms('protein and name CA')

print(f'The number of frames is {len(u.trajectory)}')

It is good to look at the SIRS trajectory to confirm that nothing is amiss.

In [ ]:
import nglview as nv

def show_sirs_trajectory():
    view = nv.show_mdanalysis(u)
    view.clear_representations()

    view.add_representation(
        "cartoon",
        selection="protein",
        color="#D0D0D0",
        opacity=1.0
    )

    view.add_representation(
        "ball+stick",
        selection="not protein and not water",
        color="red"
    )

    view.display(gui=True)
    return view
show_sirs_trajectory()

#### --> How many frames are present in the SIRS trajectory? Why is this much smaller than the total MD frames from Exercise 10?

Define secondary structure regions (transmembrane helices, loops, termini) for annotation and a helper function to add annotations to the nglview object.

In [ ]:
# UniProt P29274 topology (truncated to modeled region)
# Distinct colors for each TM, neutral intermediate colors for loops
regions = [
    (2, 7,   "N-term", "#E8DED1"),
    (8, 32,  "TM1", "#1f77b4"),   # blue
    (33, 42, "ICL1", "#E8DED1"),
    (43, 66, "TM2", "#ff7f0e"),   # orange
    (67, 77, "ECL1", "#D6C4A1"),
    (78,100, "TM3", "#2ca02c"),   # green
    (101,120,"ICL2", "#E8DED1"),
    (121,143,"TM4", "#d62728"),   # red
    (144,173,"ECL2", "#D6C4A1"),
    (174,198,"TM5", "#9467bd"),   # purple
    (199,234,"ICL3", "#E8DED1"),
    (235,258,"TM6", "#8c564b"),   # brown
    (259,266,"ECL3", "#D6C4A1"),
    (267,290,"TM7", "#e377c2"),   # pink
    (291,313,"C-term", "#E8DED1"),
]

# Adds representations and labels for regions to a nglview object
def region_visualzation(view):
    view.add_representation("cartoon", selection="all", color="#D0D0D0", opacity=1.0)
    # Color protein by UniProt regions
    for a, b, _, color in regions:
        view.add_representation("cartoon", selection=f"{a}-{b}", color=color, opacity=0.8)
    view.add_representation("licorice", selection="not protein and not water", opacity=1.0)

    # Add labels
    for a, b, label, _ in regions:
        mid = (a + b) // 2
        ca_mid = u.select_atoms(f"protein and resid {mid} and name CA")
        if ca_mid:
            view.shape.add_text((ca_mid.positions[0]).tolist(), [0, 0, 0], 5.0, label)

## Part 2 – RMSF (Root Mean Square Fluctuation) Analysis

RMSF quantifies how much each residue fluctuates around its average position over time. High RMSF values indicate flexible regions, while low RMSF values indicate rigid or well-structured regions.

Here, RMSF is calculated for **Cα atoms**, so the plot reflects backbone flexibility rather than side-chain motion.

By overlaying RMSF with [annotated secondary-structure regions (transmembrane helices, loops, termini)](https://www.uniprot.org/uniprotkb/P29274/entry#subcellular_location), we can directly relate flexibility to protein architecture.

Typical patterns include:
- **Loops and termini** showing high RMSF
- **Transmembrane helices** exhibiting lower RMSF

**Key idea:**  
RMSF tells us *where* the protein is flexible, but not *how* it moves.

In [ ]:
# Align and compute RMSF on Cα atoms
align.AlignTraj(u, u, select="protein and name CA", in_memory=True).run()

rmsf = rms.RMSF(ca).run().results.rmsf  # <- updated
resids = ca.resids

rmin, rmax = resids.min(), resids.max()
ymax = rmsf.max()

plt.figure(figsize=(12, 4))
plt.plot(resids, rmsf, lw=1, color="black")

for a, b, label, color in regions:
    if b < rmin or a > rmax:
        continue
    plt.axvspan(a, b, color=color, alpha=0.4)
    plt.text((a+b)/2, ymax*0.98, label,
             rotation=90, ha="center", va="top",
             fontsize=10, fontweight="bold")

plt.xlim(rmin, rmax)
plt.xlabel("Residue number (UniProt P29274)")
plt.ylabel("RMSF (Å)")
plt.title("A2A Adenosine Receptor Cα RMSF (UniProt-annotated)")
plt.tight_layout()
plt.show()
plt.close()

#### --> Which regions are most flexible? Do these correspond to loops or helices? Are any helices unusually mobile? Include the RMSF plot with your answer.

## Part 3 – Principal Component Analysis (PCA)

PCA identifies the **dominant collective motions** sampled during a trajectory by finding directions in coordinate space along which the system varies the most. **PC1** represents the largest-amplitude motion. **PC2, PC3, etc.** represent progressively smaller motions. Each trajectory frame is projected into this reduced coordinate space. We will focus PCA on the core of the protein.

**Key idea:**  
PCA reduces complex, many-atom motion into a few collective coordinates that describe how the protein moves.

In [ ]:
# Select Cα atoms excluding N- and C-termini
ca_core = u.select_atoms(
    "protein and name CA and resid 10:300"
)

coords = np.array([ts.positions[ca_core.indices] for ts in u.trajectory])
X = coords.reshape(coords.shape[0], -1)

pca = PCA(whiten=True)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection (core Cα atoms)")
plt.show()
plt.close()

In [ ]:
def pca_mode_magnitude(pca, mode, ca_core):
    """
    Compute per-residue magnitude of a PCA mode for Cα atoms.

    Parameters
    ----------
    pca : sklearn.decomposition.PCA
        Fitted PCA object
    mode : int
        PCA component index (0-based)
    ca_core : MDAnalysis AtomGroup
        Cα atom selection used for PCA

    Returns
    -------
    resids : np.ndarray
        Residue numbers
    magnitudes : np.ndarray
        Per-residue PCA displacement magnitudes
    """
    eig = pca.components_[mode].reshape((-1, 3))
    mag = np.linalg.norm(eig, axis=1)

    resids = ca_core.resids
    return resids, mag

def plot_pca_mode(pca, mode, ca_core, regions):
    resids, mag = pca_mode_magnitude(pca, mode, ca_core)

    rmin, rmax = resids.min(), resids.max()
    ymax = mag.max()

    plt.figure(figsize=(12, 4))
    plt.plot(resids, mag, lw=1.5, color="black")

    for a, b, label, color in regions:
        if b < rmin or a > rmax:
            continue
        b = min(b, rmax)
        plt.axvspan(a, b, color=color, alpha=0.4)
        plt.text(
            (a + b) / 2,
            ymax * 0.95,          # slightly lower than before
            label,
            rotation=90,
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold"
        )

    plt.xlim(rmin, rmax)
    plt.ylim(0, ymax * 1.1)
    plt.xlabel("Residue number (UniProt P29274)")
    plt.ylabel(f"PCA mode {mode+1} magnitude")
    plt.title(f"PCA mode {mode+1} (Cα displacement magnitude)")

    plt.subplots_adjust(top=0.88, bottom=0.15)

    plt.show()
    plt.close()

# PCA for visualization (NO whitening)
pca_raw = PCA(whiten=False)
pca_raw.fit(X)

for mode in range(5):
    plot_pca_mode(pca_raw, mode=mode, ca_core=ca_core, regions=regions)

In [ ]:
for mode in range(5):
    eig = pca_raw.components_[mode].reshape(-1, 3)
    resids = ca_core.resids
    
    ymax = np.abs(eig).max() * 1.1
    
    plt.figure(figsize=(12, 4))
    
    for i, lbl in enumerate(["x", "y", "z"]):
        plt.plot(resids, eig[:, i], lw=1.5, label=lbl)
    
    plt.axhline(0, color="black", lw=0.8, alpha=0.5)
    
    for a, b, label, color in regions:
        if b < resids.min() or a > resids.max():
            continue
        b = min(b, resids.max())
        plt.axvspan(a, b, color=color, alpha=0.3)
        plt.text((a + b) / 2, ymax * 0.95,
                 label, rotation=90, ha="center", va="top",
                 fontsize=9, fontweight="bold")
    
    plt.xlabel("Residue number (UniProt P29274)")
    plt.ylabel("PCA eigenvector component")
    plt.title(f"PCA mode {mode+1}: Cartesian components")
    plt.ylim(-ymax, ymax)
    plt.legend()
    plt.subplots_adjust(top=0.9, bottom=0.2)
    plt.show()
    plt.close()

#### --> Does the PCA distribution suggest discrete states or continuous motion? Include the PCA projection with your answer.

### Choosing the number of principal components

The scree plot shows how much variance each principal component explains. The cumulative variance plot indicates how many components are required to capture most of the motion in the trajectory.

For clustering, we aim to include enough components to capture collective motion while excluding higher-order noise. The variable `nPCs` will select the number of PCs to show on these plots.

In [ ]:
nPCs = 30

import matplotlib.pyplot as plt
import numpy as np

# Fraction of variance explained by each PC
var_ratio = pca.explained_variance_ratio_[:nPCs]
pcs = np.arange(1, len(var_ratio) + 1)

plt.figure(figsize=(6, 4))
plt.plot(pcs, var_ratio, 'o-', lw=1)
plt.xlabel("Principal Component")
plt.ylabel("Fraction of variance explained")
plt.title("PCA scree plot")
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
cum_var = np.cumsum(var_ratio)

plt.figure(figsize=(6, 4))
plt.plot(pcs, cum_var, 'o-', lw=1)
plt.axhline(0.7, color='gray', ls=':', label='70% variance')
plt.axhline(0.8, color='gray', ls='--', label='80% variance')
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative variance explained")
plt.title("Cumulative PCA variance")
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

### PCA Animation

The PCA animation exaggerates motion along a selected principal component by generating a synthetic trajectory that moves back and forth along the PCA eigenvector.

This allows us to visualize what it means, structurally, to “move along PC1” or “move along PC2”.

In [ ]:
# Helper functions for visualization
import numpy as np
from MDAnalysis.coordinates.memory import MemoryReader

def make_pca_universe(component, scale, n_frames=38):
    eig = pca.components_[component].reshape((-1, 3))
    eig /= np.linalg.norm(eig, axis=1)[:, None]

    ca_idx = ca_core.indices

    ref = u.trajectory[0].positions.copy()

    frames = []
    for t in np.linspace(0, 2*np.pi, n_frames):
        coords = ref.copy()
        coords[ca_idx] += scale * np.sin(t) * eig
        frames.append(coords)

    frames = np.asarray(frames)

    u_pca = u.copy()
    u_pca.load_new(frames, format=MemoryReader)

    return u_pca

def show_pca_animated(component=0, scale=5.0):
    u_pca = make_pca_universe(component, scale)

    view = nv.show_mdanalysis(u_pca)
    view.clear_representations()
    region_visualzation(view)

    return view

In [ ]:
# Show animation
out = widgets.Output()
def show_pca_animated_output(component=0, scale=1.0):
    out.clear_output(wait=True)
    with out:
        view = show_pca_animated(component, scale)
        display(view)       
ui = widgets.interactive(
    show_pca_animated_output,
    component=widgets.IntSlider(min=0, max=5, value=0),
    scale=widgets.FloatSlider(min=0.5, max=2.5, step=0.25, value=1)
)
display(ui)
display(out)

### Porcupine plots

In porcupine plots, PCA eigenvectors are drawn as arrows attached to atomic positions.

- Arrow direction indicates the direction of motion
- Arrow length reflects relative contribution to the mode

Porcupine plots are useful for identifying **local contributions** to motion but can be visually dense for large systems.

In [ ]:
# Helper functions for porcupine plots
def draw_pca_arrows(view, component, scale, color, stride=2):
    """
    Draw porcupine arrows for a single PCA component.

    Parameters
    ----------
    view : nglview.NGLWidget
        Active NGL view
    component : int
        PCA component index
    scale : float
        Arrow scaling factor
    color : list
        RGB color, e.g. [1, 0, 0]
    stride : int
        Downsampling factor for arrows
    """
    eig = pca.components_[component].reshape((-1, 3))
    eig /= np.linalg.norm(eig, axis=1)[:, None]

    coords = u.trajectory[0].positions[ca.indices]

    for s, d in zip(coords[::stride], eig[::stride]):
        view.shape.add_arrow(
            s.tolist(),
            (s + scale * d).tolist(),
            color,
            0.4
        )
        
def show_pca_porcupine(component1=0, component2=1, scale=10.0):
    view = nv.NGLWidget()
    view.add_component(pdb_out)
    view.clear_representations()

    region_visualzation(view)

    # Draw arrows for two PCA components using the same scale
    draw_pca_arrows(view, component1, scale, color=[1, 0, 0])  # red
    draw_pca_arrows(view, component2, scale, color=[0, 0, 1])  # blue

    view.display()
    return view

In [ ]:
# Show porcupine plot
out_porcupine = widgets.Output()
def show_pca_porcupine_output(component1=0, component2=1, scale=5.0):
    out_porcupine.clear_output(wait=True)
    with out_porcupine:
        view = show_pca_porcupine(component1, component2, scale)
        display(view)
ui_porcupine = widgets.interactive(
    show_pca_porcupine_output,
    component1=widgets.IntSlider(min=0, max=5, value=0, description="Red Arrows"),
    component2=widgets.IntSlider(min=0, max=5, value=1, description="Blue Arrows"),
    scale=widgets.FloatSlider(min=1, max=10, step=1, value=5, description="Scale")
)
display(ui_porcupine)
display(out_porcupine)

#### --> Describe the dominant motion in PC1. How does PC2 differ? Include a screenshot of the porcupine plot with your answer.

## Part 4 – Clustering in PCA Space (HDBSCAN)

Clustering asks whether the trajectory samples **distinct conformational states** rather than a single continuous basin.

Here, we perform clustering in **PCA space**, which captures the most important collective motions. We use HDBSCAN, a density-based clustering method that:
- Automatically determines the number of clusters
- Allows some frames to be labeled as noise or transitions
- Does not force all data into clusters
This makes it well suited for MD trajectories, which often exhibit continuous transitions.

**Key idea:**  
Clustering helps distinguish between continuous motion and discrete conformational states.

In [ ]:
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

X_cluster = X_pca[:, :4]
X_plot = X_pca[:, :2]

clusters = hdbscan.HDBSCAN(
    min_cluster_size=8,
    min_samples=3
).fit_predict(X_cluster)

# Relabel clusters by descending size
cluster_sizes = (pd.Series(clusters).value_counts().drop(-1, errors="ignore"))

label_map = {old: new for new, old in enumerate(cluster_sizes.index)}

clusters_relabelled = np.array([
    label_map[c] if c in label_map else -1
    for c in clusters
])
clusters = clusters_relabelled

print('Cluster index and number of samples')
print(pd.Series(clusters).value_counts())

### Coloring PCA projections by cluster identity

In [ ]:
plt.figure(figsize=(6, 5))

# noise
plt.plot(X_plot[clusters == -1, 0],
         X_plot[clusters == -1, 1],
         'o', markersize=3, color='lightgrey', label='noise')
# clusters
for k in set(clusters) - {-1}:
    m = clusters == k
    plt.plot(X_plot[m, 0], X_plot[m, 1],
             'o', markersize=6, label=f'cluster {k}')

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("HDBSCAN clustering in PCA space")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

#### --> How many clusters are identified by HDBSCAN? How large are they relative to each other? Are they well separated in PC1 vs PC2? Include the colored PC1 vs PC2 projections with your answer.

### Visualizing cluster medoids

Clustering assigns each trajectory frame to a group, but clusters themselves are abstract sets of points in PCA space. To interpret clusters structurally, we extract **medoids**.

A medoid is:
- a real trajectory frame,
- chosen as the structure closest to the cluster center in PCA space,
- representative of the conformations sampled by that cluster.

Medoids differ from averages:
- Averaged structures may be unphysical.
- Medoids correspond to actual simulated conformations.

By visualizing medoid structures, we can:
- compare conformational states,
- identify structural differences between clusters,
- connect clustering results to molecular structure.

In [ ]:
cluster_ids = np.unique(clusters)
medoid_frames = {}

for cid in cluster_ids:
    frames = np.where(clusters == cid)[0]
    centroid = X_cluster[frames].mean(axis=0)
    dists = np.linalg.norm(X_cluster[frames] - centroid, axis=1)
    medoid_frames[cid] = frames[np.argmin(dists)]

In [ ]:
from MDAnalysis.coordinates.memory import MemoryReader
import numpy as np

def make_medoid_universe(u, medoid_frames):
    # Ensure deterministic ordering
    frame_ids = sorted(medoid_frames.values())

    coords = []
    for f in frame_ids:
        u.trajectory[f]
        coords.append(u.atoms.positions.copy())

    coords = np.array(coords)  # shape: (n_medoids, n_atoms, 3)

    u_med = u.copy()
    u_med.load_new(coords, format=MemoryReader)

    return u_med, frame_ids
    
def show_cluster_medoids():
    u_med, frame_ids = make_medoid_universe(u, medoid_frames)

    view = nv.show_mdanalysis(u_med)
    view.clear_representations()

    region_visualzation(view)

    return view

show_cluster_medoids()

In this next visualization, multiple cluster medoids are superposed and displayed as separate components that can be toggled on and off using the viewer controls.

This view enables:
- direct structural comparison between clusters,
- identification of subtle differences in helix orientation or loop conformations,
- assessment of whether clusters represent meaningfully distinct states.

When comparing medoids, focus on:
- relative positions of transmembrane helices,
- changes in extracellular or intracellular loop conformations,
- differences in binding pocket geometry.

Superposed medoids provide a static complement to PCA animations, emphasizing **state differences rather than motion pathways**.

In [ ]:
import matplotlib.pyplot as plt

def cluster_color_map(clusters):
    unique = [k for k in sorted(set(clusters)) if k != -1]
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    return {k: color_cycle[i % len(color_cycle)] for i, k in enumerate(unique)}

cluster_colors = cluster_color_map(clusters)

def medoid_pca_coords(medoid_frames, X_cluster):
    """
    Return dict: cluster_id -> (PC1, PC2) coordinates of medoid.
    """
    return {
        cid: X_cluster[frame, :2]
        for cid, frame in medoid_frames.items()
        if cid != -1
    }

medoid_pca = medoid_pca_coords(medoid_frames, X_cluster)

from MDAnalysis.coordinates.memory import MemoryReader
import numpy as np

def universe_from_frame(u, frame):
    """
    Create a new MDAnalysis Universe containing exactly one frame
    taken from the original Universe `u`.

    Parameters
    ----------
    u : MDAnalysis.Universe
        Original trajectory-containing universe
    frame : int
        Frame index to extract

    Returns
    -------
    u_one : MDAnalysis.Universe
        New universe containing only the selected frame
    """
    # Move to the desired frame
    u.trajectory[frame]

    # Extract coordinates (shape: 1 x n_atoms x 3)
    coords = u.atoms.positions.copy()[None, :, :]

    # Create a copy of the topology and load a single-frame trajectory
    u_one = u.copy()
    u_one.load_new(coords, format=MemoryReader)

    return u_one

def show_superposed_medoids(max_medoids=5, showContacts=False, contact_cutoff=4.5):
    view = nv.NGLWidget()

    shown = 0
    print("Component order (for GUI toggles):")

    for cid in sorted(medoid_frames):
        if cid == -1 or shown >= max_medoids:
            continue

        frame = medoid_frames[cid]
        color = cluster_colors[cid]
        pc1, pc2 = medoid_pca[cid]

        print(
            f"  Component {shown}: "
            f"Cluster {cid}, "
            f"PC1={pc1:.2f}, PC2={pc2:.2f}"
        )

        u_one = universe_from_frame(u, frame)
        view.add_component(u_one)
        shown += 1

    for i, cid in enumerate(
        [c for c in sorted(medoid_frames) if c != -1][:shown]
    ):
        view[i].clear_representations()
        if i==0 and not showContacts:
            region_visualzation(view)
        else:
            view[i].add_representation(
                "cartoon",
                component_index=i,
                selection="all",
                color=cluster_colors[cid],
                opacity=0.9
            )
            view[i].add_representation(
                "ball+stick",
                component_index=i,
                selection="not protein and not water"
            )

        if showContacts:
            view[i].add_representation("contact", filterSele="UNK")
            d = distance_array(protein.positions, ligand.positions)
            contacting_atoms = np.any(d < contact_cutoff, axis=1)
            for res in protein[contacting_atoms].residues:
                view[i].add_representation('ball+stick', selection=f'{res.resid} and not _H')
    
    view.display(gui=True)
    return view

show_superposed_medoids(5)

#### ---> How do medoid structures differ between clusters?


## Part 5 – Protein–ligand contact visualization

This section examines which protein residues are in close contact with the ligand in a given trajectory frame.

Contacts are defined geometrically using a distance cutoff. Residues within this cutoff are visualized to highlight the **ligand interaction environment**.

When combined with clustering:
- medoids identify representative conformations,
- contact visualization reveals how binding interactions differ between states.

In [ ]:
show_superposed_medoids(5, showContacts=True)

#### --> In the medoids of each of the three most populated clusters, which protein-ligand contacts are annotated with a dashed line? Are there different patterns in each of the medoids?

### Connecting the Analyses

Each analysis in this notebook answers a different question:

- **RMSF**: Where is the protein flexible?
- **PCA**: How does the protein move collectively?
- **PCA animations / Porcupine plots**: What do these motions look like structurally?
- **Clustering**: Are there preferred conformational states?
- **Medoids**: What do those states look like as real structures?
- **Contact analysis**: How do dynamics affect ligand interactions?

Together, these methods form a coherent workflow for interpreting molecular dynamics simulations beyond raw trajectories.